# Bring your own CNN: a trained U-Net from checkpoint to Earth Engine

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb)

Takes a U-Net trained with
[segmentation-models-pytorch](https://github.com/qubvel-org/segmentation_models.pytorch) (SMP)
and runs it inside Earth Engine on your imagery:

1. check the model and find its **halo** (the border each tile needs);
2. plan the tiles;
3. upload the weights once, as Earth Engine table assets;
4. predict one test tile and compare it with PyTorch;
5. export a region.

- **Offline first.** As shipped, it uses a small demo U-Net with random weights and only builds
  the Earth Engine computations; nothing is sent.
- **Then live, step by step.** Set your checkpoint, project and input, set `RUN_LIVE = True`,
  and turn on each Earth Engine step with its own switch. Each step uses your project's Earth Engine quota (EECU time): free on a noncommercial project; a commercial project pays its plan's rate.
- **What is checked.** The building blocks agree with PyTorch at their validated settings, and
  one trained ResNet-18 U-Net, prepared this way, matched PyTorch to 1.2e-5 in Earth Engine on a
  64 × 64 input. This tiled recipe has not been run live as a whole: step 9 checks it for your
  model. Agreement with PyTorch is not accuracy.

In [ ]:
# @title 1. Install
%pip install --quiet "tensor-ee[segwater]"

## 2. Settings

- **Model:** `CHECKPOINT` is your `state_dict` file (in Colab, upload it in the **Files** panel).
  `None` = the offline demo. `ENCODER`, `BANDS` and `CLASSES` must match your training.
- **Input:** `INPUT_IMAGE` is an Earth Engine image with bands `BANDS`, already scaled as in
  training. `REGION_X`, `REGION_Y` are the region's upper-left corner in `CRS` metres; the image
  must also cover one halo around it (step 5 prints the width).
- **Earth Engine steps** run only when `RUN_LIVE` and their own switch are both `True`. You can
  change `RUN_LIVE` at any time and run from the top again.

In [ ]:
PROJECT = "YOUR_EE_PROJECT"   # your Cloud project registered for Earth Engine
RUN_LIVE = False              # True: sign in; the switches below choose the Earth Engine steps

# Your model
CHECKPOINT = None             # for example "my_unet.pth"; None = demo model, offline only
ENCODER = "resnet34"
BANDS = ("VV", "VH")          # input bands, in training order
CLASSES = 2

# Your input and region
INPUT_IMAGE = "projects/YOUR_EE_PROJECT/assets/YOUR_PREPARED_INPUT"
CRS = "EPSG:32651"
REGION_X, REGION_Y = 286030, 2640860   # upper-left corner of the region, CRS metres
PIXEL = 10                             # metres per pixel
TILE = 1024                            # region tile side, pixels
REGION_COLUMNS, REGION_ROWS = 2, 1     # region size, in tiles
TEST_TILE = 256                        # side of the test tile (step 8)

# What the live steps create
WEIGHTS_MANIFEST = "my_unet_weights.json"   # local record of the uploaded weights: keep it
TEST_ASSET = "my_unet_test_tile"
REGION_ASSET = "my_unet_region"

# Earth Engine steps
UPLOAD_WEIGHTS = False     # step 7
PREDICT_TEST_TILE = False  # step 8
COMPARE_TEST_TILE = False  # step 9
EXPORT_REGION = False      # step 10

In [ ]:
class Stop(Exception):
    """Stops the notebook with a one-line message instead of a traceback."""

    def _render_traceback_(self):
        return [f"Stopped: {self}"]


def live_step(what, switch=True):
    """True when an Earth Engine step should run."""
    if not (RUN_LIVE and switch):
        print(f"Skipped: {what} ({'RUN_LIVE' if not RUN_LIVE else 'its switch'} is False).")
        return False
    if not PROJECT or PROJECT == "YOUR_EE_PROJECT":
        raise Stop("set PROJECT to your Cloud project ID in the settings cell.")
    if CHECKPOINT is None:
        raise Stop("live steps need your trained checkpoint: set CHECKPOINT.")
    return True

In [ ]:
# @title 3. Connect: offline signatures, or your project
# @markdown Offline, Earth Engine's own test helper lets the notebook build computations without
# @markdown credentials. Switching to live undoes it, so no restart is needed.
import ee
from ee.apitestcase import ApiTestCase

if "offline" in globals():    # an earlier offline run in this session: undo it
    offline.tearDown()
    ee.data._install_cloud_api_resource = _install_cloud_api_resource
    ee.Reset()
    del offline
if not RUN_LIVE:
    _install_cloud_api_resource = ee.data._install_cloud_api_resource
    offline = ApiTestCase()
    offline.setUp()
    print("Offline: computations are built, nothing is sent.")

In [ ]:
if live_step("sign in"):
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
    print("Signed in to", PROJECT)

In [ ]:
# @title 4. Load the model
# @markdown `None`: a shallow demo U-Net with random weights. A checkpoint is loaded strictly:
# @markdown a missing or extra weight name means the settings do not match your training.
import torch
import segmentation_models_pytorch as smp

torch.set_num_threads(2)


def make_model(**changes):
    settings = dict(encoder_name=ENCODER, encoder_weights=None, in_channels=len(BANDS),
                    classes=CLASSES, activation=None)
    settings.update(changes)
    return smp.Unet(**settings).eval()


if CHECKPOINT is None:
    torch.manual_seed(20261020)
    model = make_model(encoder_depth=3, decoder_channels=(16, 8, 4))
    tile, test_tile = 64, 64      # small tiles for the demo
    print("Demo model: a shallow U-Net with random weights.")
else:
    model = make_model()
    model.load_state_dict(torch.load(CHECKPOINT, map_location="cpu", weights_only=True), strict=True)
    tile, test_tile = TILE, TEST_TILE
    print("Loaded", CHECKPOINT)

In [ ]:
# @title 5. Check support and find the halo
# @markdown The halo is how far each output pixel sees into the input. Every tile reads the
# @markdown tile plus a halo on each side, then drops it: for a ResNet-34 U-Net, 544 pixels,
# @markdown so a 1,024-pixel tile reads 2,112 × 2,112. Larger tiles waste less.
from tensor_ee import check_compatibility
from tensor_ee.receptive_field import analyze_receptive_field
from tensor_ee.smp import adapt_smp_model


def prepare(height, width):
    """The model adapted to one input size; stops unless every operation is supported."""
    shape = (1, len(BANDS), height, width)
    adapted = adapt_smp_model(model, input_shape=shape)
    report = check_compatibility(adapted, shape)
    if report.status != "supported":
        raise Stop(f"TensorEE does not support this model: {report.to_dict()['blockers'][:3]} "
                   "(notebook 02 explains reports).")
    return adapted


probe = prepare(64, 64)
support = analyze_receptive_field(probe)
halo, alignment = support.safe_halo, support.alignment
print("Supported; the prepared model equals yours on a test input:",
      probe.tensor_ee_smp_metadata["torch_equal"])
print(f"Halo {halo} px ({halo * PIXEL / 1000:g} km) per side; sizes in multiples of {alignment} px.")

In [ ]:
# @title 6. Plan the tiles
# @markdown One test tile (step 8) and the region (step 10), each with a full halo of real input.
from tensor_ee import Grid
from tensor_ee.converter import PixelWindow, plan_tiles


def plan_region(side, columns, rows):
    for name, value in (("tile side", side), ("halo", halo)):
        if value % alignment:
            raise Stop(f"the {name} ({value}) must be a multiple of {alignment} pixels.")
    padded = Grid(CRS, (PIXEL, 0, REGION_X - halo * PIXEL, 0, -PIXEL, REGION_Y + halo * PIXEL),
                  columns * side + 2 * halo, rows * side + 2 * halo)
    region = PixelWindow(halo, halo, columns * side, rows * side)
    return plan_tiles(padded, side, halo, alignment=alignment, roi=region)


def describe(plan, name):
    context = plan.tiles[0].context
    ratio = sum(t.context.pixels for t in plan.tiles) / plan.roi.pixels
    print(f"{name}: {len(plan.tiles)} tile(s) of {plan.core_size[0]} px, each reading "
          f"{context.width} x {context.height} ({ratio:.2f} input px per output px); input "
          f"{plan.global_grid.width} x {plan.global_grid.height} px from "
          f"({plan.global_grid.transform[2]}, {plan.global_grid.transform[5]}).")


test_plan = plan_region(test_tile, 1, 1)
region_plan = plan_region(tile, REGION_COLUMNS, REGION_ROWS)
describe(test_plan, "Test tile")
describe(region_plan, "Region")

In [ ]:
# @title 7. Upload the weights once
# @markdown - BatchNorm is folded into the convolutions, and the numbers go into table assets
# @markdown   `projects/PROJECT/assets/tensor_ee_weights_<random>`, one export task each (two at a time).
# @markdown - `WEIGHTS_MANIFEST` records them locally: keep it (in Colab, download it).
# @markdown - Cost not recorded; it uses your project's Earth Engine quota (EECU time): free on a noncommercial project; a commercial project pays its plan's rate. The trained ResNet-18 U-Net (14.3 M numbers) took 29 tables.
import math

from tensor_ee import effective_convolutions

parameters = effective_convolutions(probe)
values = sum(p.weights.size + (0 if p.bias is None else p.bias.size) for p in parameters)
print(f"{len(parameters)} folded convolutions, {values:,} numbers: "
      f"about {math.ceil(values * 14 / 7_000_000)} tables.")

In [ ]:
import json
import uuid
from collections import Counter
from pathlib import Path

from tensor_ee import export_weights

if live_step("upload the weights", UPLOAD_WEIGHTS):
    manifest_path = Path(WEIGHTS_MANIFEST)
    if manifest_path.exists():
        raise Stop(f"{manifest_path} exists, so the weights are already uploaded.")
    namespace = f"projects/{PROJECT}/assets/tensor_ee_weights_{uuid.uuid4().hex[:12]}"

    def progress(manifest):
        states = Counter(shard["status"] for shard in manifest["shards"])
        print(json.dumps({"status": manifest["status"], "tables": dict(states)}), flush=True)

    export_weights(parameters, namespace, manifest_path, on_update=progress)
    print("Uploaded; manifest written to", manifest_path)

In [ ]:
# @title 8. Predict the test tile
# @markdown Builds the computation (offline, with stand-in weight tables). Output bands: `score_k`
# @markdown (logits), `p_k` (probabilities), `class_index` and, for two classes, `binary_class`
# @markdown (`p_1` ≥ 0.5). Pixels missing any input band get no prediction.
import json
from pathlib import Path

from tensor_ee import MatchedWeightProvider, Raster, binary_threshold, segmentation_outputs
from tensor_ee.converter import compile_tiled_model
from tensor_ee.converter.request import deep_recursion

SCORES = tuple(f"score_{k}" for k in range(CLASSES))
PROBABILITIES = tuple(f"p_{k}" for k in range(CLASSES))


def weights_manifest():
    if RUN_LIVE and Path(WEIGHTS_MANIFEST).exists():
        return json.loads(Path(WEIGHTS_MANIFEST).read_text())
    if RUN_LIVE:
        raise Stop(f"{WEIGHTS_MANIFEST} not found: upload the weights first (step 7).")
    return {"format": "native_ee_featurecollection_weights_v2", "status": "ready",
            "layers": {p.key: {"shape": list(p.shape), "dtype": p.dtype, "bias": p.bias is not None,
                               "source": p.source, "sha256": p.sha256,
                               "assets": ["projects/YOUR_EE_PROJECT/assets/weights_not_uploaded"]}
                       for p in parameters}}


def input_image():
    if RUN_LIVE:
        return ee.Image(INPUT_IMAGE)
    return ee.Image.constant([0] * len(BANDS)).rename(list(BANDS))   # offline stand-in


def build_prediction(plan):
    context = plan.tiles[0].context
    adapted = prepare(context.height, context.width)
    if analyze_receptive_field(adapted).safe_halo != halo:
        raise Stop("the halo changed with the input size; revise the plan.")
    provider = MatchedWeightProvider(adapted, weights_manifest())
    source = input_image().select(list(BANDS)).toFloat().reproject(plan.global_grid.projection())
    valid = Raster(source.mask().reduce(ee.Reducer.min()).rename("valid"), ("valid",), plan.global_grid)
    logits = compile_tiled_model(adapted, source, BANDS, plan, weight_provider=provider)
    outputs = segmentation_outputs(logits, logit_bands=SCORES, probability_bands=PROBABILITIES,
                                   validity=valid)
    bands = [outputs.logits.image, outputs.probabilities.image, outputs.classes.image]
    if CLASSES == 2:
        bands.append(binary_threshold(outputs.probabilities, positive_class=1, threshold=0.5,
                                      output_band="binary_class").image)
    return ee.Image.cat(bands).toFloat()   # one type for export; class values stay exact


def export_task(image, plan, asset_name, description):
    grid = plan.roi_grid
    return ee.batch.Export.image.toAsset(
        image=image, description=description, assetId=f"projects/{PROJECT}/assets/{asset_name}",
        region=grid.geometry(), crs=grid.crs, crsTransform=list(grid.transform),
        maxPixels=grid.width * grid.height, pyramidingPolicy={".default": "sample"}, overwrite=False)


with deep_recursion():
    test_prediction = build_prediction(test_plan)
    test_graph_bytes = len(test_prediction.serialize().encode())
print(f"Test tile computation: {test_graph_bytes:,} bytes of Earth Engine expression.")

In [ ]:
# @title Export the test tile
# @markdown One batch export, one new image asset (the tile, not its halo). Cost not measured;
# @markdown it uses your project's Earth Engine quota (EECU time): free on a noncommercial project; a commercial project pays its plan's rate. For scale, notebook 01's 224 × 224 chip used about 65 EECU-seconds.
# @markdown If it runs out of memory or time, lower `TEST_TILE` rather than resubmitting.
import time


def wait(task, minutes=60):
    deadline = time.time() + minutes * 60
    while (state := task.status()["state"]) not in ("COMPLETED", "FAILED", "CANCELLED"):
        if time.time() > deadline:
            raise Stop("still running; follow it in the Tasks tab, or run this cell again later.")
        time.sleep(30)
    if state != "COMPLETED":
        raise Stop(f"the export ended {state}: {task.status().get('error_message', '')}")
    print("Task", task.id, "completed")


if live_step("export the test tile", PREDICT_TEST_TILE):
    test_task = export_task(test_prediction, test_plan, TEST_ASSET, "tensor_ee_test_tile")
    test_task.start()
    wait(test_task)

In [ ]:
# @title 9. Compare the test tile with PyTorch
# @markdown Downloads the tile's input (with halo) and the exported logits (2 small requests,
# @markdown no assets), runs your model locally, and reports the largest absolute difference
# @markdown (target 1e-4). The input must be complete; it is refused above 40 MB.
import numpy as np


def download(image, grid, bands):
    names = ("scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY")
    values = ee.data.computePixels({
        "expression": image.select(list(bands)), "fileFormat": "NUMPY_NDARRAY",
        "grid": {"dimensions": {"width": grid.width, "height": grid.height},
                 "affineTransform": dict(zip(names, grid.transform)), "crsCode": grid.crs}})
    return np.stack([values[band] for band in bands]).astype(np.float32)


if live_step("compare the test tile with PyTorch", COMPARE_TEST_TILE):
    padded = test_plan.global_grid
    size = padded.width * padded.height * (len(BANDS) + 1) * 4
    if size > 40_000_000:
        raise Stop(f"the input is {size / 1e6:.0f} MB; lower TEST_TILE.")
    source = ee.Image(INPUT_IMAGE).select(list(BANDS)).toFloat().reproject(padded.projection())
    valid = source.mask().reduce(ee.Reducer.min()).rename("valid")
    pixels = download(source.unmask(0).addBands(valid), padded, (*BANDS, "valid"))
    if pixels[-1].min() < 1:
        raise Stop("the input has gaps in the tile or its halo; move the region or fill them.")
    exported = download(ee.Image(f"projects/{PROJECT}/assets/{TEST_ASSET}"), test_plan.roi_grid, SCORES)

    with torch.no_grad():
        reference = model(torch.from_numpy(pixels[None, :-1].copy()))[0].numpy()
    core = reference[:, halo:halo + test_tile, halo:halo + test_tile]
    error = np.abs(exported - core)
    print(f"Largest absolute logit difference: {error.max():.3e} (target 1e-4); "
          f"mean {error.mean():.3e}; class disagreements "
          f"{int((exported.argmax(0) != core.argmax(0)).sum())} of {core[0].size} pixels.")
    print("Within the target." if error.max() <= 1e-4 else
          "Above the target: please report it, with these numbers, as an issue.")

In [ ]:
# @title 10. Export the region
# @markdown One batch export, one new image asset over the region (not its halo). Cost and
# @markdown capacity not measured at these tile sizes; it uses your project's Earth Engine quota (EECU time): free on a noncommercial project; a commercial project pays its plan's rate. Run the test tile first.
if live_step("export the region", EXPORT_REGION):
    with deep_recursion():
        region_task = export_task(build_prediction(region_plan), region_plan, REGION_ASSET,
                                  "tensor_ee_region")
        region_task.start()
    wait(region_task, minutes=240)

## Good to know

- **Evidence:** step 9 is live agreement for your model on one tile; keep its numbers.
  Accuracy is separate: evaluate the exports against your own labelled data.
- **Details:** [the guide this notebook follows](https://github.com/noelEOS/tensor-ee/blob/main/docs/own-model.md)
  covers other encoders, irregular regions and validity masks.
- **Clean up:** weight tables and images stay in your project until you delete them (Code
  Editor, **Assets** tab); keep the weight tables while you predict with them.